In VS Code, select `playground/.venv/bin/python` as the notebook kernel, then run the cells normally.

In [ ]:
from pathlib import Path
import sys

# VS Code may start the kernel either here or in the repository root.
repository_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'playground' / '__init__.py').is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError('Open this notebook from inside the MSAlign repository')
sys.path.insert(0, str(repository_root))

# MSAlign demo on custom inputs


**Notes on expected inputs:**

Provide a centroided MS/MS peak list, sorted by increasing `m/z`. Centroiding, deisotoping, noise removal, duplicate-peak merging, salt removal, and parent-molecule selection are not performed here. Intensities need not be normalized. `precursor_mz` is the measured ion m/z; do not insert it into `peaks`, because the DreaMS transform adds its own precursor pseudo-peak. Candidate strings must be valid RDKit SMILES for neutral molecules.

Inside `MSAlign`, DreaMS keeps the 100 most intense peaks, normalizes their intensities, pads the peak list, and prepends the precursor. Fused MSAlign+ also creates 0.1-Th spectrum bins, radius-2 4096-bit Morgan fingerprints, candidate exact masses, and the calibrated score combination. MolDeBERTa and DreaMS embeddings are computed on demand. Missing metadata uses learned missing/unknown encodings; a missing or unsupported adduct makes the MSAlign+ mass channels zero. See [`playground/README.md`](README.md) for the complete input contract.

In [ ]:
spectrum = {
    # Centroided [m/z, intensity] peaks. Intensities need not be normalized.
    'peaks': [
        [43.0184, 0.18],
        [77.0390, 0.12],
        [105.0335, 0.46],
        [121.0284, 1.00],
        [138.0550, 0.31],
    ],
    'precursor_mz': 181.0495,
    # Both fields below are optional and may also be None.
    'adduct': '[M+H]+',
    'collision_energy': 30.0,
}

candidate = 'CC(=O)Oc1ccccc1C(=O)O'
candidates = [candidate, 'CCO', 'CCCO', 'c1ccccc1']

In [ ]:
import os
os.environ["HF_TOKEN"] = ...

You can find the pretrained MSAlign weights at the [Zenodo record 22830464](https://zenodo.org/records/22830464)


In [ ]:
from playground import MSAlign

# Replace this path with an MSAlign+ bundle to use score fusion.
checkpoint = repository_root / 'checkpoints' / 'msalign_massspecgym_formula_seed0'
model = MSAlign(checkpoint)

In [ ]:
score = model(spectrum, candidate)
score

In [ ]:
# The returned scores follow the input SMILES order.
scores = model(spectrum, candidates)
list(zip(candidates, scores))

## Shared-space embeddings

A raw MSAlign checkpoint can also encode batches of spectra and molecules as unit-normalized vectors. Their inner product is the cosine similarity; MSAlign divides it by its learned temperature $\tau$ to produce the score:

$$\operatorname{score}(s,m)=\frac{\langle \operatorname{encode\_ms}(s),\operatorname{encode\_mol}(m)\rangle}{\tau}. $$

In [ ]:
ms_embedding = model.encode_ms([spectrum])
mol_embedding = model.encode_mol([candidate])

score_from_embeddings = ms_embedding[0].dot(mol_embedding[0]).item() / model.temperature

print(score_from_embeddings)
print(model(spectrum, candidate))

# Demo on MassSpecGym

To test MSAlign on MassSpecGym data you can directly get the spectra, candidates and splits from the 
[Zenodo record 22830464](https://zenodo.org/records/22830464)

You just need to download massspecgym.zip and unzip in data/massspecgym/

In [ ]:
import numpy as np
import json
import pandas as pd

# Load the spectras
print("MassSpecGym spectras: ")
spectra = np.load("../data/massspecgym/spectra.npy")
print(spectra.shape)

# Load the candidate map (dictionnary mapping SMILES to candidate lists)
candidate_map = json.load(open("../data/massspecgym/candidates/official_candidates_by_mass/map.json"))
print("Candidate maps: ")
key = "COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1"
ex = candidate_map[key]
print(f"Example candidates for {key}: ")
print(ex)

# Load the metadata
unique_smiles_list = pd.read_csv("../data/massspecgym/unique_smiles.csv")['smiles'].tolist()
metadata = pd.read_csv("../data/massspecgym/metadata.csv")
# Retrieve the SMILES strings for each spectra from the list of unique SMILES strings 
metadata['smiles'] = metadata['unique_smiles_idx'].apply(lambda x: unique_smiles_list[x])
# Keep only the relevant columns in the metadata
metadata = metadata[['collision_energy','adduct', 'precursor_mz', 'smiles']]
print("Metadata: ")
print(metadata.head(5))

In [ ]:
from playground import MSAlign

# Replace this path with an MSAlign+ bundle to use score fusion.
checkpoint = repository_root / 'checkpoints' / 'msalign_massspecgym_formula_seed0'
model = MSAlign(checkpoint)

In [ ]:
spectra_idx = np.random.randint(len(spectra))

# Get an input
spectra_example = {"peaks": spectra[spectra_idx],
                   "collision_energy": metadata.iloc[spectra_idx]['collision_energy'],
                   "adduct": metadata.iloc[spectra_idx]['adduct'],
                   "precursor_mz": metadata.iloc[spectra_idx]['precursor_mz']}

# Get the candidates
smiles_example = metadata.iloc[spectra_idx]['smiles']
candidates_example = candidate_map[smiles_example]
target_position = candidates_example.index(smiles_example)

# Get MSAlign scores
scores = model(spectra_example, candidates_example)

# Target rank
target_rank = scores.index(scores[target_position])
print(f"Target rank for spectra index {spectra_idx}: {target_rank}/{len(scores)}")


In [ ]:
spectra_example